# AI Recommendation Logic - Project 3

**DecodeLabs Artificial Intelligence Internship | Project 3**

This notebook builds a simple **content-based recommendation system**. The system takes user interests, converts item attributes into numerical vectors using **TF-IDF**, calculates similarity using **cosine similarity**, and displays the best matching recommendations.

## Project Requirements Covered

- Takes user interests/preferences as input
- Matches preferences using similarity logic
- Displays recommended items
- Uses recommendation concepts, pattern matching, and logic building
- Works in Google Colab even if the CSV file is missing

## Step 1: Import Required Libraries

These libraries are used for data handling, vectorization, similarity calculation, and displaying results.

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 2: Load Dataset Safely

This cell is designed to avoid `FileNotFoundError` in Google Colab.

- If `sample_recommendations.csv` exists, it loads the file.
- If the file is missing, it automatically creates a built-in sample dataset and saves it as `sample_recommendations.csv`.

So the notebook will run even if only the `.ipynb` file is uploaded to Colab.

In [10]:
DATA_PATH = Path("sample_recommendations.csv")

sample_data = [
    {"item_id": 1, "title": "Python for Beginners", "category": "Programming", "tags": "python programming beginner coding syntax variables loops functions"},
    {"item_id": 2, "title": "Machine Learning Starter", "category": "Artificial Intelligence", "tags": "machine learning python data models supervised learning algorithms"},
    {"item_id": 3, "title": "AI Automation Basics", "category": "Artificial Intelligence", "tags": "ai automation workflow tools productivity n8n integrations"},
    {"item_id": 4, "title": "Deep Learning with Neural Networks", "category": "Artificial Intelligence", "tags": "deep learning neural networks tensorflow keras cnn lstm"},
    {"item_id": 5, "title": "Data Analysis with Pandas", "category": "Data Science", "tags": "data analysis pandas numpy csv cleaning visualization python"},
    {"item_id": 6, "title": "Web Development Bootcamp", "category": "Web Development", "tags": "html css javascript frontend responsive websites"},
    {"item_id": 7, "title": "Backend Development with APIs", "category": "Web Development", "tags": "backend api flask django node databases rest server"},
    {"item_id": 8, "title": "Natural Language Processing Intro", "category": "Artificial Intelligence", "tags": "nlp text processing language models sentiment tokenization ai"},
    {"item_id": 9, "title": "Computer Vision Fundamentals", "category": "Artificial Intelligence", "tags": "computer vision images opencv object detection cnn"},
    {"item_id": 10, "title": "SQL and Database Essentials", "category": "Database", "tags": "sql database tables queries joins normalization mysql"},
    {"item_id": 11, "title": "Cloud Computing Basics", "category": "Cloud", "tags": "cloud computing deployment servers aws azure google cloud"},
    {"item_id": 12, "title": "Cybersecurity Awareness", "category": "Security", "tags": "cybersecurity privacy passwords phishing network security"},
    {"item_id": 13, "title": "Git and GitHub for Students", "category": "Software Tools", "tags": "git github version control repository commit branch project submission"},
    {"item_id": 14, "title": "Recommendation Systems Basics", "category": "Artificial Intelligence", "tags": "recommendation system similarity cosine tfidf content based filtering"},
    {"item_id": 15, "title": "Data Visualization with Matplotlib", "category": "Data Science", "tags": "data visualization charts matplotlib graphs plots python"},
    {"item_id": 16, "title": "AI Chatbot Development", "category": "Artificial Intelligence", "tags": "chatbot ai rules nlp responses intent conversation"},
    {"item_id": 17, "title": "Automation with n8n", "category": "Automation", "tags": "n8n automation workflows gmail sheets notion webhook integration"},
    {"item_id": 18, "title": "Statistics for AI", "category": "Mathematics", "tags": "statistics probability mean variance data distribution ai"},
    {"item_id": 19, "title": "Portfolio Project Building", "category": "Career", "tags": "portfolio github readme project documentation internship submission"},
    {"item_id": 20, "title": "Prompt Engineering Basics", "category": "Artificial Intelligence", "tags": "prompt engineering ai llm instructions responses automation"},
    {"item_id": 21, "title": "Business Analytics Intro", "category": "Business", "tags": "business analytics excel data decisions dashboards"},
    {"item_id": 22, "title": "Mobile App Development", "category": "App Development", "tags": "mobile app android ios flutter application development"},
    {"item_id": 23, "title": "Ethical AI and Responsible Technology", "category": "Ethics", "tags": "ethics ai privacy fairness bias responsible technology"},
    {"item_id": 24, "title": "Software Engineering Fundamentals", "category": "Software Engineering", "tags": "software engineering requirements design testing documentation lifecycle"}
]

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset loaded from file: {DATA_PATH}")
else:
    df = pd.DataFrame(sample_data)
    df.to_csv(DATA_PATH, index=False)
    print(f"Dataset file was missing, so a built-in dataset was created and saved as: {DATA_PATH}")

required_columns = {"item_id", "title", "category", "tags"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df.head()

Dataset loaded from file: sample_recommendations.csv


,item_id,title,category,tags
0,1,Python for Beginners,Programming,python programming beginner coding syntax vari...
1,2,Machine Learning Starter,Artificial Intelligence,machine learning python data models supervised...
2,3,AI Automation Basics,Artificial Intelligence,ai automation workflow tools productivity n8n ...
3,4,Deep Learning with Neural Networks,Artificial Intelligence,deep learning neural networks tensorflow keras...
4,5,Data Analysis with Pandas,Data Science,data analysis pandas numpy csv cleaning visual...


## Step 3: View Dataset Information

The dataset contains recommendation items. Each item has a title, category, and tags. The tags are the main attributes used for matching.

In [11]:
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
df[["item_id", "title", "category", "tags"]].head(10)

Dataset shape: (24, 4)
Columns: ['item_id', 'title', 'category', 'tags']


,item_id,title,category,tags
0,1,Python for Beginners,Programming,python programming beginner coding syntax vari...
1,2,Machine Learning Starter,Artificial Intelligence,machine learning python data models supervised...
2,3,AI Automation Basics,Artificial Intelligence,ai automation workflow tools productivity n8n ...
3,4,Deep Learning with Neural Networks,Artificial Intelligence,deep learning neural networks tensorflow keras...
4,5,Data Analysis with Pandas,Data Science,data analysis pandas numpy csv cleaning visual...
5,6,Web Development Bootcamp,Web Development,html css javascript frontend responsive websites
6,7,Backend Development with APIs,Web Development,backend api flask django node databases rest s...
7,8,Natural Language Processing Intro,Artificial Intelligence,nlp text processing language models sentiment ...
8,9,Computer Vision Fundamentals,Artificial Intelligence,computer vision images opencv object detection...
9,10,SQL and Database Essentials,Database,sql database tables queries joins normalizatio...


## Step 4: Create the Recommendation Function

This function performs the complete recommendation pipeline:

1. Accept user interests
2. Convert item tags and user interests into TF-IDF vectors
3. Calculate cosine similarity
4. Sort items by similarity score
5. Return the top recommendations

In [12]:
def recommend_items(user_interests, items_df, top_n=5):
    """
    Recommend items based on user interests using TF-IDF and cosine similarity.

    Parameters:
    user_interests (str or list): User interests such as 'python, ai, automation'
    items_df (DataFrame): Dataset containing item title, category, and tags
    top_n (int): Number of recommendations to return

    Returns:
    DataFrame: Top recommended items with similarity scores
    """
    if isinstance(user_interests, list):
        user_profile = " ".join(str(interest).strip() for interest in user_interests if str(interest).strip())
    else:
        user_profile = str(user_interests).replace(",", " ").strip()

    if not user_profile:
        raise ValueError("User interests cannot be empty. Please provide at least one interest.")

    clean_df = items_df.copy()
    clean_df["tags"] = clean_df["tags"].fillna("").astype(str)

    documents = clean_df["tags"].tolist() + [user_profile]

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(documents)

    item_vectors = tfidf_matrix[:-1]
    user_vector = tfidf_matrix[-1]

    similarity_scores = cosine_similarity(user_vector, item_vectors).flatten()

    result_df = clean_df.copy()
    result_df["similarity_score"] = similarity_scores
    result_df["match_percentage"] = (result_df["similarity_score"] * 100).round(2)

    result_df = result_df.sort_values(by="similarity_score", ascending=False)
    result_df = result_df.head(top_n).reset_index(drop=True)

    return result_df[["item_id", "title", "category", "tags", "similarity_score", "match_percentage"]]

print("Recommendation function created successfully.")

Recommendation function created successfully.


## Step 5: Test the Recommendation System

This test uses sample interests. You can change the interests according to your own preference.

In [13]:
user_interests = "python, ai, automation"

top_recommendations = recommend_items(user_interests, df, top_n=5)

print("User interests:", user_interests)
top_recommendations

User interests: python, ai, automation


,item_id,title,category,tags,similarity_score,match_percentage
0,20,Prompt Engineering Basics,Artificial Intelligence,prompt engineering ai llm instructions respons...,0.333143,33.31
1,3,AI Automation Basics,Artificial Intelligence,ai automation workflow tools productivity n8n ...,0.326825,32.68
2,17,Automation with n8n,Automation,n8n automation workflows gmail sheets notion w...,0.171977,17.20
3,15,Data Visualization with Matplotlib,Data Science,data visualization charts matplotlib graphs pl...,0.168553,16.86
4,5,Data Analysis with Pandas,Data Science,data analysis pandas numpy csv cleaning visual...,0.155553,15.56


## Step 6: Optional Interactive User Input

Run this cell if you want to enter your own interests manually.

Example input:

`python, machine learning, data`

In [14]:
# This cell is optional. It works in Jupyter and Google Colab.
# If you do not want interactive input, you can skip this cell.

try:
    custom_interests = input("Enter at least 3 interests separated by commas: ")
    if custom_interests.strip():
        custom_recommendations = recommend_items(custom_interests, df, top_n=5)
        print("Your interests:", custom_interests)
        display(custom_recommendations)
    else:
        print("No input entered. Using default test interests instead.")
        display(top_recommendations)
except Exception as error:
    print("Interactive input was skipped or failed:", error)
    display(top_recommendations)

Enter at least 3 interests separated by commas: Ai, Machine Learning, Deep learning
Your interests: Ai, Machine Learning, Deep learning


,item_id,title,category,tags,similarity_score,match_percentage
0,2,Machine Learning Starter,Artificial Intelligence,machine learning python data models supervised...,0.591496,59.15
1,4,Deep Learning with Neural Networks,Artificial Intelligence,deep learning neural networks tensorflow keras...,0.363881,36.39
2,20,Prompt Engineering Basics,Artificial Intelligence,prompt engineering ai llm instructions respons...,0.074114,7.41
3,3,AI Automation Basics,Artificial Intelligence,ai automation workflow tools productivity n8n ...,0.072732,7.27
4,18,Statistics for AI,Mathematics,statistics probability mean variance data dist...,0.072457,7.25


## Step 7: Save Recommendation Output

The final recommendations are saved into a CSV file named `recommendation_output.csv`.

In [15]:
OUTPUT_PATH = Path("recommendation_output.csv")
top_recommendations.to_csv(OUTPUT_PATH, index=False)
print(f"Recommendation output saved successfully as: {OUTPUT_PATH}")

Recommendation output saved successfully as: recommendation_output.csv


## Step 8: Simple Project Verification

This cell checks that the system is working correctly.

In [16]:
assert not df.empty, "Dataset should not be empty."
assert "tags" in df.columns, "Dataset must contain a tags column."
assert not top_recommendations.empty, "Recommendations should not be empty."
assert top_recommendations["similarity_score"].iloc[0] >= top_recommendations["similarity_score"].iloc[-1], "Recommendations should be sorted by score."

print("All checks passed. The recommendation system is working correctly.")

All checks passed. The recommendation system is working correctly.


## Conclusion

This project successfully implements a simple AI recommendation logic system. It uses content-based filtering, TF-IDF vectorization, and cosine similarity to match user interests with item attributes and generate ranked recommendations.

This satisfies the main Project 3 requirements: taking user input, matching preferences using similarity logic, and displaying recommended items.